In [ ]:
from binary_ext_fields.custom_field import TableField, create_field
from binary_ext_fields.rref import full_cleanup_rref, to_byte_matrix,calculate_rref, invert_pivot_rows, _partial_rref

from utils.log_helpers import get_run_log_dir, get_field_subdir, save_generation_txt, print_generation

In [ ]:
# take an example matrix

matrix_7_diag = [
    [7, 1, 1, 1, 1, 1, 1, 1, 1, 1],
    [1, 7, 1, 1, 1, 1, 1, 1, 1, 1],
    [1, 1, 7, 1, 1, 1, 1, 1, 1, 1],
    [1, 1, 1, 7, 1, 1, 1, 1, 1, 1],
    [1, 1, 1, 1, 7, 1, 1, 1, 1, 1],
    [1, 1, 1, 1, 1, 7, 1, 1, 1, 1],
    [1, 1, 1, 1, 1, 1, 7, 1, 1, 1],
    [1, 1, 1, 1, 1, 1, 1, 7, 1, 1],
    [1, 1, 1, 1, 1, 1, 1, 1, 7, 1],
    [1, 1, 1, 1, 1, 1, 1, 1, 1, 7],
]

In [ ]:
#prepare the field, 
field_3 = create_field(3)
field_3._mul


In [ ]:
#calculate partial pivot
partial_rref = _partial_rref(matrix_7_diag, field_3)
print_generation(partial_rref)

In [ ]:
# now cleanup the matrix so just a diagonal matrix is left
full_rref = full_cleanup_rref(partial_rref,field_3)
print_generation(full_rref)

In [ ]:
# now invert those rows so we get the Identity Matrix for the coefficients
identy_matrix = invert_pivot_rows(full_rref, field_3)
print_generation(identy_matrix)

In [ ]:
# now wth a matrix with data
matrix_7_diag_data = [
    [7, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 3 ,0 ,2, 6],
    [1, 7, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1 ,1 ,3, 0],
    [1, 1, 7, 1, 1, 1, 1, 1, 1, 1, 1, 3 ,3 ,2, 1],
    [1, 1, 1, 7, 1, 1, 1, 1, 1, 1, 4, 3 ,4 ,2, 2],
    [1, 1, 1, 1, 7, 1, 1, 1, 1, 1, 5, 3 ,5 ,5, 1],
    [1, 1, 1, 1, 1, 7, 1, 1, 1, 1, 0, 2 ,1 ,5, 0],
    [1, 1, 1, 1, 1, 1, 7, 1, 1, 1, 2, 3 ,4 ,5, 6],
    [1, 1, 1, 1, 1, 1, 1, 7, 1, 1, 2, 3 ,4 ,5, 6],
    [1, 1, 1, 1, 1, 1, 1, 1, 7, 1, 2, 3 ,4 ,5, 6],
    [1, 1, 1, 1, 1, 1, 1, 1, 1, 7, 2, 3 ,4 ,5, 6],
]

In [ ]:
partial_rref = _partial_rref(matrix_7_diag_data, field_3)
print_generation(partial_rref)
full_rref = full_cleanup_rref(partial_rref,field_3)
print_generation(full_rref)
identy_matrix = invert_pivot_rows(full_rref, field_3)
print_generation(identy_matrix)

In [ ]:
# Now we test where we generate our data first, and then check if the resulting matrix  is the one we generated
# need to import the tag creator

from binary_ext_fields.orthogonal_tag_creator import OrthogonalTagGenerator as OTC_custom
# and symbol generation
from binary_ext_fields.generate_symbols import generate_symbols_random, check_orth, recode


In [ ]:
#generate the data

#gen_matrix = 

# this generates until the 0 tag error is not present
accept = False
while not accept:

    data_fields = 3
    gen_size = 5
    field_size = 3

    field = create_field(field_size)

    tag_gen = OTC_custom(field)
    generation = generate_symbols_random(0,field.max_value,data_fields,gen_size)
    # new matrix with Identity matrix in the front

    gen_w_coefficients = []
    generation_size = len(generation)

    coefficients = bytearray(generation_size)


    for i, pkt in enumerate(generation):
        coefficients = bytearray(generation_size)
        coefficients[i] = 1

        gen_w_coefficients.append(coefficients.copy() + pkt.copy())


    tagged_gen = tag_gen.generate_all_tags(gen_w_coefficients)

    accept = (check_orth(field, tagged_gen)) # loop until a generation doesnt have the 0 tag error
'''
# test recode function
recoded_packets = recode(field, tagged_gen, 5)

print_generation(recoded_packets)
'''
print_generation(tagged_gen)

In [ ]:
from icecream import ic

In [ ]:
# just for checking, this should return the given mateix as is, since its already in the decoded form
partial_rref = _partial_rref(tagged_gen, field_3)
print("PARTIAL RREF")
print_generation(partial_rref)
print("\n")
full_rref = full_cleanup_rref(partial_rref,field_3)
print("CLEANED RREF")
print_generation(full_rref)
print("\n")

identy_matrix = invert_pivot_rows(full_rref, field_3)
print("Decoded Symbols")
print_generation(identy_matrix)
print("\n")


In [ ]:
# recode packets until enough independent packets are there for decoding
# check if original symbols can be recovered